# 3.4 Dynamic Prompts (adaptado a lab local)

**Qué se ve aquí:** `@dynamic_prompt` genera el **system prompt en cada llamada** a partir del runtime context (`request.runtime.context`). Aquí: el idioma de respuesta viene en `LanguageContext(user_language=...)`.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

In [ ]:
# Sin cambios
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     context_schema=LanguageContext,
#     middleware=[user_language_prompt]
# )

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [ ]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

In [ ]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

In [ ]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)